SSDTW

In [1]:
%%cython
import numpy as np
cimport numpy as np
cimport cython

import sys
import time

DTYPE_INT32 = np.int32
ctypedef np.int32_t DTYPE_INT32_t

DTYPE_FLOAT = np.float64
ctypedef np.float64_t DTYPE_FLOAT_t

cdef DTYPE_FLOAT_t MAX_FLOAT = float('inf')

# careful, without bounds checking can mess up memory - also can't use negative indices I think (like x[-1])
@cython.boundscheck(False) # turn off bounds-checking for entire function
def Segment_DP(np.ndarray[DTYPE_FLOAT_t, ndim=2] C, np.ndarray[np.int32_t, ndim=2] T):

    cdef DTYPE_INT32_t numRows = C.shape[0]
    cdef DTYPE_INT32_t numCols = C.shape[1]    
    cdef np.ndarray[np.int32_t, ndim=2] steps = np.zeros((numRows+1,numCols), dtype=np.int32)
    cdef np.ndarray[DTYPE_FLOAT_t, ndim=2] accumCost = np.ones((numRows+1, numCols), dtype=DTYPE_FLOAT) * MAX_FLOAT

    cdef unsigned int row, col
    cdef DTYPE_FLOAT_t skipCost
    cdef np.int32_t jumpStartCol
    cdef DTYPE_FLOAT_t jumpCost

    # initialize
    for row in range(numRows+1):
        for col in range(numCols):
            steps[row, col] = -1 # skip by default
    for col in range(numCols):
        accumCost[0, col] = 0 # all inf except first row
        
    # dynamic programming
    for row in range(1, numRows+1):
        for col in range(numCols):
            
            # skip transition
            if col == 0:
                skipCost = MAX_FLOAT
            else:
                skipCost = accumCost[row, col-1]
            accumCost[row, col] = skipCost
            # best step is skip by default, so don't need to assign
            
            # jump transition
            jumpStartCol = T[row-1, col]
            if jumpStartCol >= 0: # valid subsequence path
                jumpCost = accumCost[row-1, jumpStartCol] + C[row-1, col]
                if jumpCost < skipCost:
                    accumCost[row, col] = jumpCost
                    steps[row, col] = jumpStartCol

    return [accumCost, steps]

@cython.boundscheck(False) # turn off bounds-checking for entire function
def Segment_Backtrace(np.ndarray[DTYPE_FLOAT_t, ndim=2] accumCost, np.ndarray[np.int32_t, ndim=2] steps):

    cdef np.uint32_t numRows = accumCost.shape[0]
    cdef np.uint32_t numCols = accumCost.shape[1]
    cdef np.uint32_t curRow = numRows - 1
    cdef np.uint32_t curCol = numCols - 1
    cdef np.int32_t jump
    cdef np.ndarray[np.uint32_t, ndim=1] path = np.zeros(numRows-1, dtype=np.uint32)
    cdef np.uint32_t pathElems = 0

    while curRow > 0:
        if accumCost[curRow, curCol] == MAX_FLOAT:
            print('A path is not possible')
            break

        jump = steps[curRow, curCol]
        if jump < 0: # skip
            curCol = curCol - 1
        else: # jump
            path[pathElems] = curCol
            pathElems = pathElems + 1
            curRow = curRow - 1
            curCol = jump

    return path[::-1]

@cython.boundscheck(False) # turn off bounds-checking for entire function
def calc_Tseg(np.ndarray[DTYPE_FLOAT_t, ndim=2] accumCost, np.ndarray[np.uint32_t, ndim=2] stepsForCost, parameter):
    '''

    Parameter should have: 'dn', 'dm'
    '''

    cdef np.ndarray[unsigned int, ndim=1] dn
    cdef np.ndarray[unsigned int, ndim=1] dm
    cdef np.uint32_t numRows = accumCost.shape[0]
    cdef np.uint32_t numCols = accumCost.shape[1]
    cdef np.ndarray[np.int32_t, ndim=1] startLocs = np.zeros(numCols, dtype=np.int32)
    cdef np.uint32_t endCol
    cdef np.uint32_t curRow
    cdef np.uint32_t curCol
    cdef np.uint32_t curStepIndex

    # get step transitions
    if ('dn'  in parameter.keys()):
        dn = parameter['dn']
    else:
        dn = np.array([1, 1, 0], dtype=DTYPE_INT32)
    if 'dm'  in parameter.keys():
        dm = parameter['dm']
    else:
        dm = np.array([1, 0, 1], dtype=DTYPE_INT32)

    # backtrace from every location
    for endCol in range(numCols):
        curCol = endCol
        curRow = numRows - 1
        while curRow > 0:
            if accumCost[curRow, curCol] == MAX_FLOAT: # no valid path
                startLocs[curCol] = -1
                break

            curStepIndex = stepsForCost[curRow, curCol]
            curRow = curRow - dn[curStepIndex]
            curCol = curCol - dm[curStepIndex]
            if curRow == 0:
                startLocs[endCol] = curCol
                
    return startLocs

class bcolors:
    HEADER = '\033[95m'
    OKBLUE = '\033[94m'
    OKGREEN = '\033[92m'
    WARNING = '\033[93m'
    FAIL = '\033[91m'
    ENDC = '\033[0m'
    BOLD = '\033[1m'
    UNDERLINE = '\033[4m'


UsageError: Cell magic `%%cython` not found.


In [5]:
# def alignSSDTW(featfile1, featfile2, steps, weights, downsample, outfile = None, profile = False):
#     # compute cost matrix
#     F1data = np.load(featfile1) # 88 x N
#     F1 = F1data['roll']
#     F2data = np.load(featfile2) # 88 x M
#     F2 = F2data['roll']
#     segment_boundaries = F2data['measure_boundaries']

#     swap = (F1.shape[1] > F2.shape[1])
#     if swap:
#         F1, F2 = F2, F1 # make the shorter sequence the query

#     if len(segment_boundaries) <= 1: # if no valid segments, use whole matrix
#         segment_boundaries = np.array([0, F2.shape[1]])

#     if max(F1.shape[1], F2.shape[1]) / min(F1.shape[1], F2.shape[1]) >= 2: # no valid path possible
#         if outfile:
#             pickle.dump(None, open(outfile, 'wb'))
#         return None
    
#     times = []
#     times.append(time.time())
#     C = 1 - F1[:,0::downsample].T @ F2[:,0::downsample] # cos distance metric
#     times.append(time.time())
    
#     # run subseqDTW on chunks
#     numSegments = len(segment_boundaries) - 1
#     dn = steps[:,0].astype(np.uint32)
#     dm = steps[:,1].astype(np.uint32)
#     dw = weights
#     params1 = {'dn': dn, 'dm': dm, 'dw': dw, 'SubSequence': True}
#     Dparts = []
#     Bparts = []
#     val_seg = []
#     for i in range(numSegments):
#         start_idx = segment_boundaries[i]
#         end_idx = segment_boundaries[i+1]

#         start_idx = max(0, min(start_idx, C.shape[0]-1)) # clamping
#         end_idx = max(start_idx+1, min(end_idx, C.shape[0])) # clamping

#         if end_idx -  start_idx < 2: # need at least 2 frames for any of valid steps
#             continue

#         Cpart = C[start_idx:end_idx, :]
#         [D, B] = DTW_Cost_To_AccumCostAndSteps(Cpart, params1)
#         Dparts.append(D)
#         Bparts.append(B)
#         val_seg.append(i)

#     if len(val_seg) == 0 :
#         return None, C

#     times.append(time.time())
    
#     # construct Cseg, Tseg
#     Cseg = np.zeros((numSegments, F2.shape[1]))
#     Tseg = np.zeros((numSegments, F2.shape[1]), dtype=np.int32)

#     for i, Dpart in enumerate(Dparts):
#         Cseg[i,:] = Dpart[-1,:]
#         Tseg[i,:] = calc_Tseg(Dpart, Bparts[i], params1)
#     times.append(time.time())
    
#     # segment-level DP
#     [Dseg, Bseg] = Segment_DP(Cseg, Tseg)
#     times.append(time.time())
#     segmentEndIdxs = Segment_Backtrace(Dseg, Bseg)
#     times.append(time.time())
    
#     # backtrace on chunks
#     wps = []
#     for i, endidx in enumerate(segmentEndIdxs):
#         seg  = val_seg[i]
#         params2 = {'dn': dn, 'dm': dm, 'dw': dw, 'SubSequence': True, 'startCol': endidx}
#         [wpchunk, _, _] = DTW_GetPath(Dparts[i], Bparts[i], params2)
#         wpchunk[0,:] = wpchunk[0,:] + segment_boundaries[seg]  # account for relative offset
#         wps.append(wpchunk.copy())
#     wp_merged = np.hstack(wps)
#     times.append(time.time())
    
#     if swap:
#         wp_merged = np.flipud(wp_merged) # undo swap
    
#     if outfile:
#         pickle.dump(wp_merged, open(outfile, 'wb'))
    
#     if profile:
#         return wp_merged, np.diff(times)
#     else:
#         return wp_merged, C

In [ ]:
def alignSSDTW(featfile1, featfile2, steps, weights, downsample, frame_indices=None, outfile=None, profile=False):
    """
    Strictly Segmented DTW alignment with frame indices for segmentation
    """
    
    import pickle
    F1data = np.load(featfile1) # 88 x N
    F1 = F1data['roll']
    F2data = np.load(featfile2) # 88 x M
    F2 = F2data['roll']
    
    swap = (F1.shape[1] > F2.shape[1])
    if swap:
        F1, F2 = F2, F1 # make the shorter sequence the query
    
    # Determine segmentation boundaries
    if frame_indices is not None:
        # Convert to numpy array if needed
        if isinstance(frame_indices, list):
            segment_boundaries = np.array(frame_indices, dtype=np.int32)
        else:
            segment_boundaries = frame_indices
            
        # Ensure boundaries include start and end points
        if 0 not in segment_boundaries:
            segment_boundaries = np.insert(segment_boundaries, 0, 0)
        if F2.shape[1] not in segment_boundaries:
            segment_boundaries = np.append(segment_boundaries, F2.shape[1])
            
    else:
        segment_boundaries = np.array([0, F2.shape[1]]) # use whole matrix

    if max(F1.shape[1], F2.shape[1]) / min(F1.shape[1], F2.shape[1]) >= 2: # check if matricies are too different in length (no valid path possible)
        if outfile:
            pickle.dump(None, open(outfile, 'wb'))
        return None, None
    
    times = []
    times.append(time.time())

    # Jaccard distance
    F1_sampled = F1[:,::downsample].T
    F2_sampled = F2[:,::downsample].T
    C = cdist(F1_sampled, F2_sampled, metric='jaccard')
    times.append(time.time())
    
    # Run DTW on each segment
    numSegments = len(segment_boundaries) - 1
    dn = steps[:,0].astype(np.uint32)
    dm = steps[:,1].astype(np.uint32)
    dw = weights
    params1 = {'dn': dn, 'dm': dm, 'dw': dw, 'SubSequence': True}
    
    Dparts = []
    Bparts = []
    val_seg = []
    
    for i in range(numSegments):
        start_idx = segment_boundaries[i]
        end_idx = segment_boundaries[i+1]
        
        # Ensure valid segment boundaries
        start_idx = max(0, min(start_idx, C.shape[0]-1))
        end_idx = max(start_idx+1, min(end_idx, C.shape[0]))
        
        if end_idx - start_idx < 2:
            print(f"{bcolors.WARNING}Segment {i} ({start_idx}:{end_idx}) too small. Skipping.{bcolors.ENDC}")
            continue
            
        # Extract segment from cost matrix
        Cpart = C[start_idx:end_idx, :]
        print(f"Processing segment {i} ({start_idx}:{end_idx}), shape: {Cpart.shape}")
        
        # Run standard DTW on this segment
        [D, B] = DTW_Cost_To_AccumCostAndSteps(Cpart, params1)
        
        Dparts.append(D)
        Bparts.append(B)
        val_seg.append(i)
    
    # Check if there's valid segments
    if len(val_seg) == 0:
        if outfile:
            pickle.dump(None, open(outfile, 'wb'))
        return None, C
        
    times.append(time.time())
    

    # construct Cseg, Tseg
    Cseg = np.zeros((numSegments, F2.shape[1]))
    Tseg = np.zeros((numSegments, F2.shape[1]), dtype=np.int32)
    
    for i, (j, Dpart) in enumerate(zip(val_seg, Dparts)):
        Cseg[i,:] = Dpart[-1,:]
        Tseg[i,:] = calc_Tseg(Dpart, Bparts[i], params1)
        
    times.append(time.time())
    
    # segment-level DP
    [Dseg, Bseg] = Segment_DP(Cseg, Tseg)
    times.append(time.time())
    
    # Get segment end indices from segment-level backtrace
    segmentEndIdxs = Segment_Backtrace(Dseg, Bseg)
    times.append(time.time())
    
    # backtrace on chunks
    wps = []
    for i, endidx in enumerate(segmentEndIdxs):
        seg  = val_seg[i]
        params2 = {'dn': dn, 'dm': dm, 'dw': dw, 'SubSequence': True, 'startCol': endidx}
        [wpchunk, _, _] = DTW_GetPath(Dparts[i], Bparts[i], params2)
        wpchunk[0,:] = wpchunk[0,:] + segment_boundaries[seg]  # account for relative offset
        wps.append(wpchunk.copy())
        
        wps.append(wpchunk.copy())
    
    wp_merged = np.hstack(wps)
    times.append(time.time())
    
    if swap:
        wp_merged = np.flipud(wp_merged)
    
    if outfile:
        pickle.dump(wp_merged, open(outfile, 'wb'))
    
    if profile:
        return wp_merged, np.diff(times)
    else:
        return wp_merged, C